# Day 5 · 두 서비스를 라우팅하고 평가로 배포를 결정하기

화면을 따라 실행하되, 결과를 자동 게시하지 않습니다. 모든 외부 쓰기는 dry-run과 사람 승인을 먼저 거칩니다.

In [1]:
# 최초 1회 설치. 이미 설치했다면 빠르게 완료됩니다.
%pip install -q -r ../../requirements-day1.txt
# STT 실습을 실제 음성으로 실행할 때만 다음 줄의 주석을 해제합니다.
# %pip install -q -r ../../requirements-stt-optional.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print({"workspace": str(ROOT), "python": sys.version.split()[0]})

{'workspace': '/Users/sungjae-cha/sungjae-cha/llm-agent-and-workflow-automation', 'python': '3.12.12'}


## 1. 입력 종류를 명시해 회의와 코드 리뷰 서비스로 라우팅합니다

In [3]:
from src.course_services.service_router import route_service_request

meeting = route_service_request(
    input_kind="meeting_transcript",
    source_path=ROOT / "data/meeting_sample_ko.txt",
    workspace_root=ROOT,
)
review = route_service_request(
    input_kind="code_diff",
    source_path=ROOT / "data/day3_review_cases/unsafe_pr.diff",
    workspace_root=ROOT,
)
print(json.dumps({"meeting": meeting["service"], "review": review["service"]}, ensure_ascii=False, indent=2))

{
  "meeting": "meeting",
  "review": "code_review"
}


## 2. Golden finding과 현재 결과를 같은 key로 비교합니다

In [4]:
from src.course_services.eval_service import evaluate_review_findings, release_gate

expected = json.loads((ROOT / "data/day5_eval/golden_review_findings.json").read_text(encoding="utf-8"))
metrics = evaluate_review_findings(review["result"]["findings"], expected)
gate = release_gate(review_metrics=metrics, safety_passed=True, latency_seconds=0.2)
print(json.dumps({"metrics": metrics, "release_gate": gate}, ensure_ascii=False, indent=2))

{
  "metrics": {
    "true_positive": 3,
    "false_positive": 0,
    "false_negative": 0,
    "precision": 1.0,
    "recall": 1.0,
    "f1": 1.0
  },
  "release_gate": {
    "decision": "READY",
    "reasons": [],
    "human_release_required": true
  }
}


## 3. HOLD를 일부러 만들어 운영 판단을 연습합니다

In [5]:
hold = release_gate(
    review_metrics={"precision": 0.7, "recall": 0.6},
    safety_passed=False,
    latency_seconds=42.0,
)
print(json.dumps(hold, ensure_ascii=False, indent=2))
assert hold["decision"] == "HOLD"

{
  "decision": "HOLD",
  "reasons": [
    "RECALL_BELOW_0_8",
    "PRECISION_BELOW_0_8",
    "SAFETY_CHECK_FAILED",
    "LATENCY_BUDGET_EXCEEDED"
  ],
  "human_release_required": true
}


## 완료 확인

- Day 5 결과 JSON을 확인했습니다.
- 실패 경로가 traceback 대신 `error_code`로 남는지 확인했습니다.
- 외부 쓰기와 자동 메일이 발생하지 않았음을 확인했습니다.
- 변경한 코드는 diff와 test 결과를 사람이 검토합니다.